Set up the VIIRS NDVI data to be processed using Google Earth Engine

Author: Tayler LaSharr;
Date: 10 Sept 2023


In [ ]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime

## if you need to authenticate GEE use this code
## ee.Authenticate('your-GEE-project')

## initialize the Earth Engine module
ee.Initialize()

In [ ]:
## load in the VIIRS data and the MODIS snow data
viirs_ndvi = ee.ImageCollection('NASA/VIIRS/002/VNP13A1') \
                .filterDate('2018-01-01', '2023-12-31') \
                .select('NDVI')

viirs_evi = ee.ImageCollection('NASA/VIIRS/002/VNP13A1') \
                .filterDate('2018-01-01', '2023-12-31') \
                .select('EVI')

modis_snow = ee.ImageCollection('MODIS/006/MOD10A1') \
                .filterDate('2018-01-01', '2023-12-31') \
                .select('NDSI_Snow_Cover')

In [ ]:
## now run the smoothing process on the data following Bischof et al. 2002's methods

## define a function to use a moving median filter to reduce noise
def apply_median_filter(image):
    
    ## Apply focal median filter with a radius of 3 pixels
    smoothed_image = image.focal_median(radius=3, units='pixels')
    
    ## Retain the 'system:time_start' property
    return smoothed_image.set('system:time_start', image.get('system:time_start'))


## now use the function on the NDVI image collection
smoothed_viirs_ndvi = viirs_ndvi.map(apply_median_filter)

In [ ]:
## now we will fix snow in the NDVI data. For now, I am going to shift slightly from 
## the Bischof methods because I don't want to overestimate green-up for desert systems

## to do this, first we will define a function to mask snow cover,
## this is a step to help us understand how snow plays into the curves
## especially important for desert systems, right now I set it to mask 
## cells with more than 35% snow cover

def apply_snow_mask(image, snow_image):
    snow_mask = ee.Algorithms.If(
        snow_image,
        snow_image.lte(35),  ## if the snow cover is less than or equal to 35%
        ee.Image(1)          ## default mask (all values included if no corresponding snow image)
    )
    return image.updateMask(snow_mask)

## now mask the NDVI data with more than 35% snow cover
viirs_ndvi_without_snow = clipped_viirs_ndvi.map(
    lambda image: apply_snow_mask(image, modis_snow.filterDate(image.date(), image.date().advance(1, 'day')).first())
)    